## Market Question
How can we estimate the real estate prices in Turin based on property features and geographic location?  
Goal: help investors, buyers, and companies better understand the city’s real estate market.

## What We Will Do
- Collect real data from online property listings.
- Clean and organize the data in SQL databases.
- Perform exploratory data analysis and create interactive visualizations with Tableau.
- Build a machine learning model to predict property prices.
- Develop a final dashboard and provide clear insights to support business decisions.

First of all i used a Google extention called Web Scraper. It took me 1 hour to scrape 1388 rows of datas. The output is in the Excel file named ImmobiliareFinale.xlsx in the data folder. Next i'll show you the first few lines of the file: 

In [3]:
import pandas as pd

df = pd.read_excel('data/ImmobiliareFinale.xlsx')
df.head()

,web_scraper_order,web_scraper_start_url,pagination,title,data,data2,photo,photo2,image,image2,...,elevator,balcony,last_update,Product_3_Energy_Class,characteristics,energy_class,energy_consumption,image_1,text_15,text_16
0,1774637625-1,https://www.immobiliare.it/vendita-case/torino...,https://www.immobiliare.it/vendita-case/torino...,"Bilocale via Michele Berrino 19, Barriera di L...",77000.0,premium,NaN,NaN,https://pic.im-cdn.it/imagenoresize/1115579022...,https://pwm.im-cdn.it/image/1870347433/xxs-c.jpg,...,Ascensore,Balcone,Annuncio aggiornato il 26/03/2026,"Rif: BER17 - Cerchi un appartamento luminoso, ...",Esposizione esternaFibra otticaInfissi esterni...,F,216.58 kWh/m² annoF,https://pwm.im-cdn.it/image/1870347433/cover-m...,NaN,"AbitazionePiano2Superficie67,0 m²Coefficiente1..."
1,1774637627-2,https://www.immobiliare.it/vendita-case/torino...,https://www.immobiliare.it/vendita-case/torino...,"Appartamento via Tommaso Villa 73, Mirafiori S...",189000.0,premium,NaN,NaN,https://pic.im-cdn.it/imagenoresize/1588645549...,https://pwm.im-cdn.it/image/1869903391/xxs-c.jpg,...,Ascensore,Balcone,Annuncio aggiornato il 12/03/2026,METODOCASA IMMOBILIARE\n\nPropone in vendita I...,Esposizione doppiaFibra otticaImpianto di alla...,E,≥ 175 kWh/m² annoE,https://pwm.im-cdn.it/image/1869903391/cover-m...,NaN,"AbitazionePiano6Superficie120,0 m²Coefficiente..."
2,1774637629-3,https://www.immobiliare.it/vendita-case/torino...,https://www.immobiliare.it/vendita-case/torino...,"Appartamento via degli Ulivi 17, Falchera, Torino",95000.0,premium,NaN,NaN,https://pic.im-cdn.it/imagenoresize/1588645549...,https://pwm.im-cdn.it/image/1870309373/xxs-c.jpg,...,Ascensore,Balcone,Annuncio aggiornato il 11/03/2026,METODOCASA IMMOBILIARE\n\nPropone in vendita E...,Armadio a muroEsposizione doppiaFibra otticaIn...,E,≥ 175 kWh/m² annoE,https://pwm.im-cdn.it/image/1870309373/cover-m...,NaN,"AbitazionePiano6Superficie105,0 m²Coefficiente..."
3,1774637632-4,https://www.immobiliare.it/vendita-case/torino...,https://www.immobiliare.it/vendita-case/torino...,"Appartamento via Bibiana 107, Borgo Vittoria, ...",259000.0,premium,NaN,NaN,https://pic.im-cdn.it/imagenoresize/1860778981...,https://pwm.im-cdn.it/image/1870304835/xxs-c.jpg,...,Ascensore,Balcone,Annuncio aggiornato il 19/02/2026,ZONA BORGO VITTORIA\nVENDESI – ULTIMO PIANO CO...,Esposizione doppiaFibra otticaImpianto di alla...,D,78.44 kWh/m² annoD,https://pwm.im-cdn.it/image/1870304835/cover-m...,NaN,"AbitazionePiano4Superficie186,0 m²Coefficiente..."
4,1774637634-5,https://www.immobiliare.it/vendita-case/torino...,https://www.immobiliare.it/vendita-case/torino...,"Bilocale via Gian Francesco Re 114, Parella, T...",147000.0,premium,NaN,NaN,https://pic.im-cdn.it/imagenoresize/1088608097...,https://pwm.im-cdn.it/image/1875691533/xxs-c.jpg,...,Ascensore,Balcone,Annuncio aggiornato il 26/02/2026,Torino (TO) - zona Parella: in zona residenzia...,Esposizione esternaFibra otticaInfissi esterni...,D,≥ 175 kWh/m² annoD,https://pwm.im-cdn.it/image/1875691533/cover-m...,NaN,"AbitazionePiano2Superficie59,0 m²Coefficiente1..."


## Data Cleaning

We have some work to do with data cleaning.

---

### Step 1: Drop useless features

First, we remove columns that were generated by the web scraper but carry no analytical value:

| Column | Reason |
|--------|--------|
| `web_scraper_order` | Internal scraper index |
| `web_scraper_start_url` | Scraper start URL, not needed |
| `pagination` | Pagination URL, not needed |
| `data2` | Duplicate of price field |
| `photo` | Duplicate of image field |
| `photo2` | Duplicate of image field |
| `image` | Low resolution thumbnail |
| `image2` | Low resolution thumbnail |

---

### Step 2: Check and remove duplicates

Since the dataset was scraped across multiple pages, the same listing may appear more than once.  
We use `item_page_link` as the unique identifier for each listing, as every ad on immobiliare.it has a distinct URL.  
We first check how many duplicates exist, then remove them keeping only the first occurrence.

In [3]:
import pandas as pd

df = pd.read_excel('data/ImmobiliareFinale_featureclean.xlsx')
print(f'Righe totali: {len(df)}')
print(f'Duplicati: {df.duplicated(subset="item_page_link").sum()}')
print(f'Righe uniche: {df["item_page_link"].nunique()}')

Righe totali: 1388
Duplicati: 0
Righe uniche: 1388


#### Excel Confirmation

As a confirmation that there are no duplicates, the following Excel formula was used 
and it confirmed the Python result:

```excel
=COUNT.SE(C:C, immobiliare_finale[@[item_page_link]]) - 1
```

The formula counts how many times each `item_page_link` appears in the column and subtracts 1 
(the row itself). A result of **0** for all rows confirms that every listing is unique.

>  **All 1,388 rows are unique, no duplicates found!**

Below is a representation of the first rows of the new dataset:

In [8]:
import pandas as pd

df = pd.read_excel('data/ImmobiliareFinale_featureclean.xlsx')
df.head()

,title,data,item_page_title,item_page_link,title_1,data_1,location,Property_1_Size,rooms,bathrooms,...,elevator,balcony,last_update,Product_3_Energy_Class,characteristics,energy_class,energy_consumption,image_1,text_15,text_16
0,"Bilocale via Michele Berrino 19, Barriera di L...",77000.0,"Bilocale via Michele Berrino 19, Barriera di L...",https://www.immobiliare.it/annunci/126843897/,"Bilocale via Michele Berrino 19, Barriera di L...",€ 77.000,TorinoBarriera di LanzoVia Michele Berrino 19,67 m²,2 locali,1 bagno,...,Ascensore,Balcone,Annuncio aggiornato il 26/03/2026,"Rif: BER17 - Cerchi un appartamento luminoso, ...",Esposizione esternaFibra otticaInfissi esterni...,F,216.58 kWh/m² annoF,https://pwm.im-cdn.it/image/1870347433/cover-m...,NaN,"AbitazionePiano2Superficie67,0 m²Coefficiente1..."
1,"Appartamento via Tommaso Villa 73, Mirafiori S...",189000.0,"Appartamento via Tommaso Villa 73, Mirafiori S...",https://www.immobiliare.it/annunci/126824775/,"Appartamento via Tommaso Villa 73, Mirafiori S...",€ 189.000,TorinoMirafiori Sud - Onorato ViglianiVia Tomm...,120 m²,5 locali,1 bagno,...,Ascensore,Balcone,Annuncio aggiornato il 12/03/2026,METODOCASA IMMOBILIARE\n\nPropone in vendita I...,Esposizione doppiaFibra otticaImpianto di alla...,E,≥ 175 kWh/m² annoE,https://pwm.im-cdn.it/image/1869903391/cover-m...,NaN,"AbitazionePiano6Superficie120,0 m²Coefficiente..."
2,"Appartamento via degli Ulivi 17, Falchera, Torino",95000.0,"Appartamento via degli Ulivi 17, Falchera, Torino",https://www.immobiliare.it/annunci/126842237/,"Appartamento via degli Ulivi 17, Falchera, Torino",€ 95.000,"TorinoFalcheraVia degli Ulivi, 17",105 m²,5 locali,1 bagno,...,Ascensore,Balcone,Annuncio aggiornato il 11/03/2026,METODOCASA IMMOBILIARE\n\nPropone in vendita E...,Armadio a muroEsposizione doppiaFibra otticaIn...,E,≥ 175 kWh/m² annoE,https://pwm.im-cdn.it/image/1870309373/cover-m...,NaN,"AbitazionePiano6Superficie105,0 m²Coefficiente..."
3,"Appartamento via Bibiana 107, Borgo Vittoria, ...",259000.0,"Appartamento via Bibiana 107, Borgo Vittoria, ...",https://www.immobiliare.it/annunci/126842063/,"Appartamento via Bibiana 107, Borgo Vittoria, ...",€ 259.000,"TorinoBorgo VittoriaVia Bibiana, 107",186 m²,5+ locali,3 bagni,...,Ascensore,Balcone,Annuncio aggiornato il 19/02/2026,ZONA BORGO VITTORIA\nVENDESI – ULTIMO PIANO CO...,Esposizione doppiaFibra otticaImpianto di alla...,D,78.44 kWh/m² annoD,https://pwm.im-cdn.it/image/1870304835/cover-m...,NaN,"AbitazionePiano4Superficie186,0 m²Coefficiente..."
4,"Bilocale via Gian Francesco Re 114, Parella, T...",147000.0,"Bilocale via Gian Francesco Re 114, Parella, T...",https://www.immobiliare.it/annunci/126828147/,"Bilocale via Gian Francesco Re 114, Parella, T...",€ 147.000,"TorinoParellaVia Gian Francesco Re, 114",59 m²,2 locali,1 bagno,...,Ascensore,Balcone,Annuncio aggiornato il 26/02/2026,Torino (TO) - zona Parella: in zona residenzia...,Esposizione esternaFibra otticaInfissi esterni...,D,≥ 175 kWh/m² annoD,https://pwm.im-cdn.it/image/1875691533/cover-m...,NaN,"AbitazionePiano2Superficie59,0 m²Coefficiente1..."


In [14]:
import re

def parse_title(title):
    parts = [p.strip() for p in title.split(',')]
    

    city = parts[-1] if len(parts) >= 1 else None

    quartiere = parts[-2] if len(parts) >= 2 else None
    
    address_part = ','.join(parts[:-2]).strip() if len(parts) >= 3 else None
    
    street = None
    civico = None
    tipo = None
    
    if address_part:
        match = re.search(
            r'(via|corso|piazza|largo|viale|strada|vicolo|borgata|lungo po|lungo dora|c\.so|p\.za)',
            address_part, re.IGNORECASE
        )
        if match:
            tipo = address_part[:match.start()].strip()
            street_full = address_part[match.start():].strip()
            num_match = re.search(r'[\s,]+(\d+[\w/]*)$', street_full)
            if num_match:
                civico = num_match.group(1)
                street = street_full[:num_match.start()].strip().rstrip(',')
            else:
                street = street_full
        else:
            tipo = address_part

    return pd.Series({
        'tipo_immobile': tipo,
        'via': street,
        'civico': civico,
        'quartiere': quartiere,
        'citta': city
    })

# Applica la funzione e aggiungi le colonne al dataframe
df[['tipo_immobile', 'via', 'civico', 'quartiere', 'citta']] = df['title'].apply(parse_title)
df.to_excel('data/ImmobiliareFinale_address.xlsx', sheet_name='clean', index=False)
df.head()

,title,data,item_page_title,item_page_link,title_1,data_1,location,Property_1_Size,rooms,bathrooms,...,energy_class,energy_consumption,image_1,text_15,text_16,tipo_immobile,via,civico,quartiere,citta
0,"Bilocale via Michele Berrino 19, Barriera di L...",77000.0,"Bilocale via Michele Berrino 19, Barriera di L...",https://www.immobiliare.it/annunci/126843897/,"Bilocale via Michele Berrino 19, Barriera di L...",€ 77.000,TorinoBarriera di LanzoVia Michele Berrino 19,67 m²,2 locali,1 bagno,...,F,216.58 kWh/m² annoF,https://pwm.im-cdn.it/image/1870347433/cover-m...,NaN,"AbitazionePiano2Superficie67,0 m²Coefficiente1...",Bilocale,via Michele Berrino,19,Barriera di Lanzo,Torino
1,"Appartamento via Tommaso Villa 73, Mirafiori S...",189000.0,"Appartamento via Tommaso Villa 73, Mirafiori S...",https://www.immobiliare.it/annunci/126824775/,"Appartamento via Tommaso Villa 73, Mirafiori S...",€ 189.000,TorinoMirafiori Sud - Onorato ViglianiVia Tomm...,120 m²,5 locali,1 bagno,...,E,≥ 175 kWh/m² annoE,https://pwm.im-cdn.it/image/1869903391/cover-m...,NaN,"AbitazionePiano6Superficie120,0 m²Coefficiente...",Appartamento,via Tommaso Villa,73,Mirafiori Sud - Onorato Vigliani,Torino
2,"Appartamento via degli Ulivi 17, Falchera, Torino",95000.0,"Appartamento via degli Ulivi 17, Falchera, Torino",https://www.immobiliare.it/annunci/126842237/,"Appartamento via degli Ulivi 17, Falchera, Torino",€ 95.000,"TorinoFalcheraVia degli Ulivi, 17",105 m²,5 locali,1 bagno,...,E,≥ 175 kWh/m² annoE,https://pwm.im-cdn.it/image/1870309373/cover-m...,NaN,"AbitazionePiano6Superficie105,0 m²Coefficiente...",Appartamento,via degli Ulivi,17,Falchera,Torino
3,"Appartamento via Bibiana 107, Borgo Vittoria, ...",259000.0,"Appartamento via Bibiana 107, Borgo Vittoria, ...",https://www.immobiliare.it/annunci/126842063/,"Appartamento via Bibiana 107, Borgo Vittoria, ...",€ 259.000,"TorinoBorgo VittoriaVia Bibiana, 107",186 m²,5+ locali,3 bagni,...,D,78.44 kWh/m² annoD,https://pwm.im-cdn.it/image/1870304835/cover-m...,NaN,"AbitazionePiano4Superficie186,0 m²Coefficiente...",Appartamento,via Bibiana,107,Borgo Vittoria,Torino
4,"Bilocale via Gian Francesco Re 114, Parella, T...",147000.0,"Bilocale via Gian Francesco Re 114, Parella, T...",https://www.immobiliare.it/annunci/126828147/,"Bilocale via Gian Francesco Re 114, Parella, T...",€ 147.000,"TorinoParellaVia Gian Francesco Re, 114",59 m²,2 locali,1 bagno,...,D,≥ 175 kWh/m² annoD,https://pwm.im-cdn.it/image/1875691533/cover-m...,NaN,"AbitazionePiano2Superficie59,0 m²Coefficiente1...",Bilocale,via Gian Francesco Re,114,Parella,Torino


In [15]:
missing = df[['tipo_immobile', 'via', 'civico', 'quartiere', 'citta']].isna().sum()
print("Valori mancanti per colonna:")
print(missing)
print()
print(f"Totale righe: {len(df)}")
print()

Valori mancanti per colonna:
tipo_immobile      8
via               45
civico           162
quartiere          2
citta              0
dtype: int64

Totale righe: 1388

